# 01 — RQ1: Statistical association of care discordance and readmission
**RQ1:** How is care-intensity discordance associated with 30-day readmission after accounting for recorded patient characteristics and clinical complexity?

Analysis unit: hospital encounter; repeated encounters are clustered by patient. Use the held-out test partition for the prespecified association model, with discordance generated by the training-only expected-care benchmark. This is retrospective observational association, not evidence of under-treatment or causal effects. The four signed residuals are measured in training residual SD units (not necessarily centred).

In [ ]:
# Shared input setup (run from the project or statistical_analysis folder).
if (!requireNamespace("jsonlite", quietly=TRUE)) stop("Install jsonlite in R first; see README.")
root <- normalizePath(getwd(), winslash="/", mustWork=TRUE)
while (!file.exists(file.path(root,"statistical_analysis/private_exports/manifest.json"))) {
  parent <- dirname(root)
  if (identical(parent,root)) stop("Run 00_export_for_R.ipynb first.")
  root <- parent
}
input <- file.path(root,"statistical_analysis/private_exports")
output <- file.path(root,"statistical_analysis/outputs")
dir.create(output,recursive=TRUE,showWarnings=FALSE)
meta <- jsonlite::fromJSON(file.path(input,"manifest.json"))
read_split <- function(name) {
  d <- read.csv(file.path(input,paste0(name,".csv")), na.strings=c(""), check.names=FALSE,
                stringsAsFactors=FALSE)
  stopifnot(nrow(d)==meta$rows[[name]], all(d[[meta$target]] %in% c(0,1)),
            !anyNA(d$patient_cluster), all(is.finite(as.matrix(d[meta$discordance]))))
  d
}
train <- read_split("train")
test <- read_split("test")
y <- test[[meta$target]]
stopifnot(length(unique(y))==2)
cat("Training encounters:", nrow(train), "| Test encounters:",nrow(test),"\n")

## Descriptive summary
Report encounter counts, means, medians and interquartile ranges for each signed discordance dimension by readmission. The boxplots are descriptive; they do not adjust for confounders.

In [ ]:
desc <- do.call(rbind,lapply(meta$discordance,function(v) {
  do.call(rbind,lapply(0:1,function(g) {
    z <- test[y==g,v]
    data.frame(dimension=v,readmitted=g,n=length(z),mean=mean(z),sd=sd(z),median=median(z),
               Q1=unname(quantile(z,.25)),Q3=unname(quantile(z,.75)))
  }))
}))
print(desc); write.csv(desc,file.path(output,"rq1_descriptive.csv"),row.names=FALSE)
plot_disc <- function() {
  old <- par(mfrow=c(2,2),mar=c(4,4,3,1)); on.exit(par(old))
  for (v in meta$discordance) {
    boxplot(test[[v]]~factor(y,levels=0:1,labels=c("No","Yes")),main=sub("discordance_","",v),
            xlab="30-day readmission",ylab="Signed residual / training SD",outline=FALSE)
    abline(h=0,lty=2)
  }
}
plot_disc()
png(file.path(output,"rq1_discordance_boxplots.png"),width=1400,height=1000,res=140); plot_disc(); dev.off()

## Adjusted logistic regression
Fit a joint model of all four discordance dimensions and the clinical covariates specified by the feature contract. Do not additionally adjust for the observed care variables from which these residuals are constructed: the estimand is conditional association with discordance given clinical covariates, not an independent causal effect. Numerical missingness: training median and a missingness indicator where needed. Categorical levels with fewer than 50 training encounters and unseen levels are pooled into `Other_or_rare`; this is a prespecified pragmatic rule. Missing categorical values are explicit before pooling. Patient-cluster robust standard errors account for repeated encounters, but not uncertainty in the previously fitted benchmark. If separation or singularity occurs, stop and document it rather than quietly discarding results.

In [ ]:
if (!requireNamespace("sandwich",quietly=TRUE)) stop("Install sandwich; see README.")
analysis <- test[c("patient_cluster",meta$target,meta$discordance)]
controls <- character()
for (v in meta$clinical_numeric) {
  a <- as.numeric(train[[v]]); b <- as.numeric(test[[v]])
  if (!any(is.finite(a))) stop(paste("No training values for",v))
  med <- median(a[is.finite(a)])
  missing <- !is.finite(b)
  b[missing] <- med
  if (length(unique(b))>1) {analysis[[v]] <- b; controls <- c(controls,v)}
  if (any(missing) && any(!missing)) {
    name <- paste0(v,"_missing"); analysis[[name]] <- as.integer(missing); controls <- c(controls,name)
  }
}
for (v in meta$clinical_categorical) {
  a <- as.character(train[[v]]); b <- as.character(test[[v]])
  a[is.na(a)] <- "Missing"; b[is.na(b)] <- "Missing"
  counts <- table(a); keep <- names(counts[counts>=50])
  b[!b %in% keep] <- "Other_or_rare"
  f <- factor(b)
  if (nlevels(f)>1) {
    # Most frequent observed group as reference; changes interpretation of controls only.
    f <- relevel(f,ref=names(sort(table(f),decreasing=TRUE))[1])
    analysis[[v]] <- f; controls <- c(controls,v)
  }
}
form <- reformulate(c(controls,meta$discordance),response=meta$target)
fit_warnings <- character()
fit <- withCallingHandlers(glm(form,data=analysis,family=binomial(),control=glm.control(maxit=100)),
   warning=function(w) {fit_warnings <<- c(fit_warnings,conditionMessage(w)); invokeRestart("muffleWarning")})
if (length(fit_warnings)) print(fit_warnings)
if (!fit$converged || anyNA(coef(fit)) || any(!is.finite(coef(fit))) ||
    any(grepl("0 or 1|converge",fit_warnings))) stop("Unstable or singular model; inspect sparse categories and report before interpretation.")
V <- sandwich::vcovCL(fit,cluster=analysis$patient_cluster,type="HC1")
if (any(!is.finite(V)) || any(diag(V)<=0)) stop("Invalid cluster-robust covariance.")
terms <- meta$discordance
beta <- coef(fit)[terms]; se <- sqrt(diag(V)[terms]); z <- beta/se
p <- 2*pnorm(-abs(z))
result <- data.frame(dimension=terms,log_OR=unname(beta),robust_SE=unname(se),OR=unname(exp(beta)),
                     CI_low=unname(exp(beta-1.96*se)),CI_high=unname(exp(beta+1.96*se)),
                     p_value=unname(p),p_Holm=p.adjust(unname(p),method="holm"))
if (any(!is.finite(as.matrix(result[-1])))) stop("Nonfinite estimates; investigate model stability.")
print(result); write.csv(result,file.path(output,"rq1_adjusted_odds_ratios.csv"),row.names=FALSE)
cat("Encounters:",nobs(fit),"| Patient clusters:",length(unique(analysis$patient_cluster)),"\n")

## Primary joint hypothesis test and forest plot
H0: all four adjusted discordance coefficients are zero. Use a cluster-robust Wald chi-square test at alpha 0.05. Individual 95% CIs are nominal; Holm-adjusted p-values address four component tests. Odds ratios are per one training residual SD increase, not per unit of original care. An OR above 1 means higher adjusted odds, not a higher absolute probability by that amount.

In [ ]:
W <- as.numeric(crossprod(beta,solve(V[terms,terms,drop=FALSE],beta)))
stopifnot(is.finite(W),W>=0)
joint <- data.frame(Wald_chisq=W,df=length(terms),p_value=pchisq(W,length(terms),lower.tail=FALSE),
                    encounters=nobs(fit),patients=length(unique(analysis$patient_cluster)))
print(joint); write.csv(joint,file.path(output,"rq1_joint_wald.csv"),row.names=FALSE)
forest <- function() {
  old <- par(mar=c(5,13,3,1)); on.exit(par(old))
  pos <- rev(seq_len(nrow(result)))
  plot(result$OR,pos,log="x",xlim=range(c(1,result$CI_low,result$CI_high)),ylim=c(.5,nrow(result)+.5),
       yaxt="n",ylab="",xlab="Adjusted odds ratio (95% CI)",pch=19,main="RQ1: Patient-cluster robust estimates")
  segments(result$CI_low,pos,result$CI_high,pos); abline(v=1,lty=2)
  axis(2,at=pos,labels=sub("discordance_","",result$dimension),las=1,cex.axis=.85)
}
forest(); png(file.path(output,"rq1_forest.png"),width=1500,height=800,res=140); forest(); dev.off()

## Interpretation to complete after execution
Report the joint Wald result, four adjusted ORs with CIs and Holm p-values, and whether effects are practically meaningful. A non-significant result is not proof of no relationship. Recorded diagnoses and utilisation do not fully measure severity; reverse causation, unmeasured confounding and historical care patterns remain possible. The test is conditional on the trained benchmark and on eligible recorded encounters. No clinical advice follows from these associations.